# Stage 1 — Raw Page Structure Inspection


## 1. Priority Classification

In [2]:
import json
import re
from pathlib import Path
from collections import defaultdict

RAW_DIR = Path('../../data/raw_scraped_pages')
CLASSIF = Path('../../docs/url_classification.json')

# --- load pages (page_*.json only; footer.json is handled separately in §2) ---
# raw files have two schemas:
#   schema A (newer spider): page_url / page_title / all_text_markdown / sections
#   schema B (older spider): source_url / text_markdown / links / raw_html
def get_url(page):
    return page.get('page_url') or page.get('source_url', '')

def get_title(page):
    return page.get('page_title', '')

pages = []
schema_counts = {'A': 0, 'B': 0}
for fpath in sorted(RAW_DIR.glob('page_*.json')):
    with open(fpath, encoding='utf-8') as f:
        p = json.load(f)
    pages.append(p)
    if 'page_url' in p:
        schema_counts['A'] += 1
    else:
        schema_counts['B'] += 1

print(f'Loaded {len(pages)} raw pages  (page_*.json only, footer.json excluded)')
print(f'  Schema A (page_url / page_title / all_text_markdown): {schema_counts["A"]}')
print(f'  Schema B (source_url / text_markdown / raw_html):     {schema_counts["B"]}')

# --- load classification ---
with open(CLASSIF, encoding='utf-8') as f:
    classif = json.load(f)

entries      = classif['entries']
url_patterns = classif['url_patterns']
print(f'\nClassification loaded: {len(entries)} entries, {len(url_patterns)} url_patterns')

Loaded 176 raw pages  (page_*.json only, footer.json excluded)
  Schema A (page_url / page_title / all_text_markdown): 176
  Schema B (source_url / text_markdown / raw_html):     0

Classification loaded: 68 entries, 19 url_patterns


In [3]:
# --- URL normalizer: raw pages may lack trailing slash ---
def norm(url: str) -> str:
    return url.rstrip('/')

# Build lookup: norm(url) -> entry
entry_lookup = {}
for e in entries:
    entry_lookup[norm(e['url'])] = e
    entry_lookup.setdefault(norm(e['canonical_url']), e)

def classify_url(url: str) -> dict:
    key = norm(url)
    if key in entry_lookup:
        return entry_lookup[key]
    for pat in url_patterns:
        if re.match(pat['regex'], url):
            return pat
    return {'priority': 'skip', 'url': url}

# --- classify every page ---
PRIORITY_ORDER = ['must', 'dsi-general', 'optional', 'alias', 'skip']
classified = defaultdict(list)   # priority -> list of dicts

for p in pages:
    url   = get_url(p)
    title = get_title(p)
    result   = classify_url(url)
    priority = result.get('priority', 'unlisted')
    classified[priority].append({'url': url, 'title': title})

print('Classification summary:')
for pri in PRIORITY_ORDER:
    print(f'  {pri:12s}: {len(classified[pri]):3d} pages')

Classification summary:
  must        :  14 pages
  dsi-general :   7 pages
  optional    :  10 pages
  alias       :   0 pages
  skip        : 145 pages


In [4]:
# --- Print each priority group in full ---
for priority in PRIORITY_ORDER:
    group = classified[priority]
    if not group:
        continue
    print(f'\n{"="*72}')
    print(f'  [{priority.upper()}]  ({len(group)} pages)')
    print(f'{"="*72}')
    for item in sorted(group, key=lambda x: x['url']):
        print(f'  {item["url"]}')


  [MUST]  (14 pages)
  https://datascience.uchicago.edu/education/masters-programs/in-person-program
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-project-archive
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-projects
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/career-outcomes
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/course-progressions
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/events-deadlines
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply
  https://datascience.uchicago.edu/education/masters-programs/ms-in-appl

In [5]:
# --- Gap check: sitemap must/should entries MISSING from raw ---
raw_url_norms = {norm(get_url(p)) for p in pages}

print('Sitemap must/should entries MISSING from raw_scraped_pages:\n')
missing = []
for e in entries:
    if e['priority'] in ('must', 'should'):
        if norm(e['url']) not in raw_url_norms and norm(e['canonical_url']) not in raw_url_norms:
            missing.append(e)

if missing:
    for e in missing:
        print(f'  [{e["priority"]}]  {e["url"]}')
else:
    print('  None — all must/should entries are present in raw data.')

Sitemap must/should entries MISSING from raw_scraped_pages:

  None — all must/should entries are present in raw data.


## 2. Header & Footer Classification

In [6]:
FOOTER_PATH = RAW_DIR.parent / 'raw_scraped_pages' / 'footer.json'

with open(FOOTER_PATH, encoding='utf-8') as f:
    footer = json.load(f)

footer_hrefs = [link['href'] for link in footer.get('links', []) if link.get('href')]
print(f'Footer hrefs: {len(footer_hrefs)} total')

# Compare against the 177 raw scraped pages
raw_url_norms = {norm(get_url(p)) for p in pages}

footer_classified = []
for href in footer_hrefs:
    tag = 'dsi-general' if norm(href) in raw_url_norms else 'footer-only'
    footer_classified.append({'url': href, 'tag': tag})

dsi_general_count = sum(1 for x in footer_classified if x['tag'] == 'dsi-general')
footer_only_count  = sum(1 for x in footer_classified if x['tag'] == 'footer-only')

print(f'  In raw pages  → dsi-general : {dsi_general_count}')
print(f'  Not in raw    → footer-only : {footer_only_count}')
print()
for item in sorted(footer_classified, key=lambda x: x['url']):
    print(f'  [{item["tag"]:12s}]  {item["url"]}')

Footer hrefs: 21 total
  In raw pages  → dsi-general : 12
  Not in raw    → footer-only : 9

  [footer-only ]  https://accessibility.uchicago.edu
  [footer-only ]  https://bsky.app/profile/dsi-uchicago.bsky.social
  [dsi-general ]  https://datascience.uchicago.edu
  [dsi-general ]  https://datascience.uchicago.edu/about
  [dsi-general ]  https://datascience.uchicago.edu/about/contact
  [dsi-general ]  https://datascience.uchicago.edu/about/jobs
  [dsi-general ]  https://datascience.uchicago.edu/about/leadership-staff
  [dsi-general ]  https://datascience.uchicago.edu/education
  [dsi-general ]  https://datascience.uchicago.edu/news-events/events
  [dsi-general ]  https://datascience.uchicago.edu/news-events/news
  [dsi-general ]  https://datascience.uchicago.edu/newsletter-archive
  [dsi-general ]  https://datascience.uchicago.edu/nondiscrimination-statement
  [dsi-general ]  https://datascience.uchicago.edu/outreach
  [dsi-general ]  https://datascience.uchicago.edu/research
  [footer

## 3. Generate Bottom Table

### 3.1  Build url_classes from classified (§1) + footer_classified (§2)

In [7]:
# Base: classified['must/dsi-general/optional/skip']  (§1 result)
# Promote: footer hrefs that ARE in raw pages (tag='dsi-general' in §2)
#          → if currently skip/not in top-3, move to dsi-general
# Add:     footer hrefs NOT in raw pages (tag='footer-only' in §2)
#          → footer-only bucket

# Build norm→priority map from §1 classified
_norm_to_pri = {}
for pri in ('must', 'dsi-general', 'optional', 'skip'):
    for item in classified[pri]:
        _norm_to_pri[norm(item['url'])] = pri

url_classes = {
    'must':        [item['url'] for item in classified['must']],
    'dsi-general': [item['url'] for item in classified['dsi-general']],
    'optional':    [item['url'] for item in classified['optional']],
    'footer-only': [],
}

for fc in footer_classified:
    n   = norm(fc['url'])
    tag = fc['tag']   # 'dsi-general' = in raw pages | 'footer-only' = not in raw
    if tag == 'footer-only':
        url_classes['footer-only'].append(fc['url'])
    else:
        # in raw pages — promote to dsi-general if currently skip or unclassified
        current = _norm_to_pri.get(n, 'skip')
        if current not in ('must', 'dsi-general', 'optional'):
            url_classes['dsi-general'].append(fc['url'])

# ── Print counts ──
print('url_classes counts:')
for pri in ('must', 'dsi-general', 'optional', 'footer-only'):
    print(f'  {pri:12s}: {len(url_classes[pri])}')

print('\nmust URLs:')
for u in sorted(url_classes['must']):
    print(f'  {u}')

print('\ndsi-general URLs:')
for u in sorted(url_classes['dsi-general']):
    print(f'  {u}')

print('\noptional(ads-captone research) URLs:')
for u in sorted(url_classes['optional']):
    print(f'  {u}')

print('\nfooter-only URLs:')
for u in sorted(url_classes['footer-only']):
    print(f'  {u}')

url_classes counts:
  must        : 14
  dsi-general : 14
  optional    : 10
  footer-only : 9

must URLs:
  https://datascience.uchicago.edu/education/masters-programs/in-person-program
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-project-archive
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-projects
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/career-outcomes
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/course-progressions
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/events-deadlines
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/

In [8]:
optional_urls = [entry["url"] for entry in entries if entry.get("priority") == "optional"]

import copy
url_class_reference = copy.deepcopy(url_classes)
url_class_reference["optional"] = optional_urls

import json
class_path = Path.cwd().parent.parent / "docs" / "url_class_reference.json"
with open(class_path, "w", encoding="utf-8") as f:
    json.dump(url_class_reference, f, indent=2, ensure_ascii=False)

### 3.2  Build records (must/dsi-general/optional) + footer dict

In [9]:
KEEP_PRIORITIES = {'must', 'dsi-general', 'optional'}

# norm(url) → priority from url_classes
_cls_norm_to_pri = {}
for pri, urls in url_classes.items():
    for u in urls:
        _cls_norm_to_pri[norm(u)] = pri

# Build norm(url) → filename in one pass (page_*.json only)
_norm_to_file = {}
for fpath in sorted(RAW_DIR.glob('page_*.json')):
    with open(fpath, encoding='utf-8') as f:
        _p = json.load(f)
    _norm_to_file[norm(get_url(_p))] = fpath.name

records = []
for p in pages:
    url      = get_url(p)
    priority = _cls_norm_to_pri.get(norm(url))
    if priority not in KEEP_PRIORITIES:
        continue
    entry = entry_lookup.get(norm(url), {})
    records.append({
        'source_file':       _norm_to_file.get(norm(url), ''),
        'url':               url,
        'canonical_url':     entry.get('canonical_url', url),
        'priority':          priority,
        'depth':             entry.get('depth', None),
        'special_handling':  entry.get('special_handling', []),
        'page_title':        p.get('page_title', ''),
        'meta_description':  p.get('meta_description', ''),
        'breadcrumbs':       p.get('breadcrumbs', []),
        'scraped_at':        p.get('scraped_at', ''),
        'all_text_markdown': p.get('all_text_markdown') or p.get('text_markdown', ''),
        'sections':          p.get('sections', None),
    })


# ── Summary ──
print(f'records total: {len(records)}')
pri_counts = defaultdict(int)
for r in records:
    pri_counts[r['priority']] += 1
for pri in ('must', 'dsi-general', 'optional'):
    print(f'  {pri:12s}: {pri_counts[pri]}')

tag_counts = defaultdict(int)
for r in records:
    for tag in r['special_handling']:
        tag_counts[tag] += 1
print('\nspecial_handling tag counts:')
if tag_counts:
    for tag, cnt in sorted(tag_counts.items(), key=lambda x: -x[1]):
        files = [r['source_file'] for r in records if tag in r['special_handling']]
        print(f'  {tag:45s}: {cnt}  ({", ".join(files)})')
else:
    print('  (none)')



records total: 38
  must        : 14
  dsi-general : 14
  optional    : 10

special_handling tag counts:
  verify_accordion_expansion                   : 3  (page_146.json, page_148.json, page_159.json)
  flag_for_periodic_rescrape                   : 1  (page_164.json)
  exclude_registration_links                   : 1  (page_164.json)
  exclude_image_embedded_content               : 1  (page_165.json)


In [10]:
from urllib.parse import urlparse

def _platform_name(href: str) -> str:
    """Derive platform name from hostname when link text is empty.
    https://x.com/...          -> 'X'
    https://www.linkedin.com/  -> 'LinkedIn'
    https://bsky.app/...       -> 'Bsky'
    """
    host = urlparse(href).hostname or ''
    host = host.removeprefix('www.')   # strip leading www.
    name = host.split('.')[0]          # take part before first dot
    return name.capitalize()

# ── footer dict: links from footer.json ──
footer_links = [
    {
        'text': link.get('text') or _platform_name(link['href']),
        'href': link['href'],
    }
    for link in footer.get('links', [])
    if link.get('href')
]

print(f'footer_links: {len(footer_links)} entries')
for fl in footer_links:
    print(f'  {fl["text"]!r:30s}  {fl["href"]}')

footer_links: 21 entries
  'DSI'                           https://datascience.uchicago.edu
  'About'                         https://datascience.uchicago.edu/about
  'People'                        https://datascience.uchicago.edu/about/leadership-staff
  'Research'                      https://datascience.uchicago.edu/research
  'Education'                     https://datascience.uchicago.edu/education
  'Outreach'                      https://datascience.uchicago.edu/outreach
  'News'                          https://datascience.uchicago.edu/news-events/news
  'Events'                        https://datascience.uchicago.edu/news-events/events
  'Jobs'                          https://datascience.uchicago.edu/about/jobs
  'Newsletter Archive'            https://datascience.uchicago.edu/newsletter-archive
  'Physical Sciences Division'    https://physicalsciences.uchicago.edu
  'Nondiscrimination Statement'   https://datascience.uchicago.edu/nondiscrimination-statement
  'Accessibilit

## 4. Clean Universal Artifacts

In [34]:
# Deep-copy records so the raw foundational wide table is preserved
import copy
cleaned_records = copy.deepcopy(records)

### 4.1 `must` batch

In [47]:
# -- must batch --
must_idx = [i for i, r in enumerate(cleaned_records) if r["priority"] == "must"]
print(f"=== must ({len(must_idx)} pages) ===")
for i in must_idx:
    print(f"{i:<3} | {cleaned_records[i]['source_file']:<15} | {cleaned_records[i]['url']}")

=== must (14 pages) ===
14  | page_042.json   | https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science
15  | page_145.json   | https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply
16  | page_146.json   | https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs
17  | page_147.json   | https://datascience.uchicago.edu/education/masters-programs/online-program
18  | page_148.json   | https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/course-progressions
19  | page_149.json   | https://datascience.uchicago.edu/education/masters-programs/in-person-program
20  | page_159.json   | https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/instructors-staff
21  | page_160.json   | https://datascience.uchicago.edu/explore-the-ms-ads-campus
22  | page_161.json   | https://datascience.uchicago.edu/education/masters-progra

In [ ]:
hp_idx = next(i for i, r in enumerate(cleaned_records) if r['source_file'] == 'page_042.json')
pp_idx = next(i for i, r in enumerate(cleaned_records) if r['source_file'] == 'page_159.json')
print(hp_idx,pp_idx)

14 20 16 18


In [ ]:
# reconstruct the section 
cleaned_records[hp_idx]['structured_sections'] = [
                                                {'section_index': sec['section_index'],
                                                'text_markdown': sec['text_markdown']}
                                                for sec in cleaned_records[hp_idx]['sections']
                                            ]

def content_to_structured_sections(sections):
    """Split a record's sections into heading-based sub-sections.
    Each heading item starts a new sub-section: text_markdown = '## {heading text}'.
    Non-heading items append their text with a leading newline.
    Items before the first heading are collected into a preamble sub-section.
    """
    structured = []
    idx = 0
    current_md = None

    for sec in sections:
        for item in (sec.get('content') or []):
            text = item.get('text', '')
            if item.get('type') == 'heading':
                if current_md is not None:
                    structured.append({'section_index': idx, 'text_markdown': current_md})
                    idx += 1
                current_md = f"## {text}"
            else:
                if current_md is None:
                    current_md = text   # preamble before first heading
                else:
                    current_md += f"\n{text}"

    if current_md is not None:
        structured.append({'section_index': idx, 'text_markdown': current_md})

    return structured

for i in must_idx:
    if i == hp_idx:
        continue
    cleaned_records[i]['structured_sections'] = content_to_structured_sections(
        cleaned_records[i]['sections']
    )

# Verification
print("structured_sections written for must (excl. idx hp_idx):")
for i in must_idx:
    ss = cleaned_records[i]['structured_sections']
    print(f"  [{i}] {cleaned_records[i]['source_file']}: {len(ss)} sub-sections")
    for s in ss[:4]:
        preview = s['text_markdown'][:90].replace('\n', '↵')
        print(f"       [{s['section_index']}] {preview}")

structured_sections written for must (excl. idx 14):
  [14] page_042.json: 9 sub-sections
       [0] The University of Chicago’s MS in Applied Data Science program equips you with in-demand e
       [1] ## Programs↵Choose from full- and part-time options in our In-Person and Online programs. 
       [2] ## Apply Today!↵The application for entry is open! Prospective Online and In-Person studen
       [3] ## How to Apply↵You have questions; we have answers. Whether you are just beginning your s
  [15] page_145.json: 17 sub-sections
       [0] ## [MS in Applied Data Science](https://datascience.uchicago.edu/education/masters-program
       [1] ## Follow↵https://www.linkedin.com/company/dsi-uchicago↵https://www.youtube.com/@UChicagoD
       [2] ## How to Apply
       [3] ## Master’s in Applied Data Science Application Requirements↵The [application portal](http
  [16] page_146.json: 4 sub-sections
       [0] ## [MS in Applied Data Science](https://datascience.uchicago.edu/education/masters-

In [77]:
# Drop section_index 0 and 1 (nav header + Follow sidebar) for must records except idx 14 and 20
for i in must_idx:
    if i in (hp_idx, pp_idx):
        continue
    cleaned_records[i]['structured_sections'] = [
        s for s in cleaned_records[i]['structured_sections']
        if s['section_index'] not in (0, 1)
    ]

In [78]:
# Print first 90 chars of each section for all must records
for i in must_idx:
    ss = cleaned_records[i].get('structured_sections', [])
    print(f"\n[{i}] {cleaned_records[i]['source_file']}  ({len(ss)} sections), {cleaned_records[i]['url']}")
    for s in ss:
        preview = s['text_markdown'][:90].replace('\n', '↵')
        print(f"  [{s['section_index']}] {preview}")


[14] page_042.json  (9 sections), https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science
  [0] The University of Chicago’s MS in Applied Data Science program equips you with in-demand e
  [1] ## Programs↵Choose from full- and part-time options in our In-Person and Online programs. 
  [2] ## Apply Today!↵The application for entry is open! Prospective Online and In-Person studen
  [3] ## How to Apply↵You have questions; we have answers. Whether you are just beginning your s
  [4] ## Industry Leading Faculty↵UChicago’s Master’s in Applied Data Science program recruits i
  [5] ## Data in Action - Capstone Projects↵Whether you are early in your career or more advance
  [6] ## How UChicago’s MS in Applied Data Science Prepares Future Innovators↵Learn how the Univ
  [7] ## Start Your Application↵The [application](https://apply-psd.uchicago.edu/apply)to be con
  [8] ## Related News, Insights, and Past Events↵## University of Chicago Announces Partnership 

[15]

### 4.2 other priority class ———— pending decision on whether to process

In [14]:
# ── dsi-general batch ──
dsi_idx = [i for i, r in enumerate(cleaned_records) if r['priority'] == 'dsi-general']
print(f"=== dsi-general ({len(dsi_idx)} pages) ===")
# display([cleaned_records[i] for i in dsi_idx])

=== dsi-general (14 pages) ===


In [15]:
# -- optional batch --
opt_idx = [i for i, r in enumerate(cleaned_records) if r["priority"] == "optional"]
print(f"=== optional ({len(opt_idx)} pages) ===")
# display([cleaned_records[i] for i in opt_idx])

=== optional (10 pages) ===


## 5. Per special_handling Cleaning

In [82]:
tag_idx = defaultdict(list)
for i in must_idx:
    for tag in cleaned_records[i]['special_handling']:
        tag_idx[tag].append(i)

print('##--------- special_handling tags (must only) ---------##')
if tag_idx:
    for tag, indices in sorted(tag_idx.items(), key=lambda x: -len(x[1])):
        print(f'  {tag:45s}: {indices}')
else:
    print('  (none)')

##--------- special_handling tags (must only) ---------##
  verify_accordion_expansion                   : [16, 18, 20]
  flag_for_periodic_rescrape                   : [25]
  exclude_registration_links                   : [25]
  exclude_image_embedded_content               : [26]


In [102]:
event_idx = next(i for i, r in enumerate(cleaned_records) if r['source_file'] == 'page_164.json')
career_idx = next(i for i, r in enumerate(cleaned_records) if r['source_file'] == 'page_165.json')

for r in cleaned_records:
    r['Note'] = ''

print(f'event_idx={event_idx}, career_idx={career_idx}')
print(f'Note field initialized for {len(cleaned_records)} records.')

event_idx=25, career_idx=26
Note field initialized for 38 records.


### 5.1 Warning for periodic scraping

In [103]:
scraped = cleaned_records[event_idx]['scraped_at']
cleaned_records[event_idx]['Note'] = (
    f"This page contains time-sensitive information (events and application deadlines). "
    f"The content was scraped on {scraped}. "
    "Currency of the information cannot be guaranteed — please verify current details on the official website."
)
print(cleaned_records[event_idx]['Note'])

This page contains time-sensitive information (events and application deadlines). The content was scraped on 2026-04-19T01:43:22.233687+00:00. Currency of the information cannot be guaranteed — please verify current details on the official website.


### 5.2 Warning for image content is not supported

In [104]:
cleaned_records[career_idx]['Note'] = (
    "This model relies solely on webpage text content. "
    "This page contains image-embedded content (e.g., employer logos, career outcome charts) "
    "that cannot be interpreted from images. Exercise caution when asking about such visuals."
)
print(cleaned_records[career_idx]['Note'])

This model relies solely on webpage text content. This page contains image-embedded content (e.g., employer logos, career outcome charts) that cannot be interpreted from images. Exercise caution when asking about such visuals.


## 6. PII Redaction

- Redact personal email addresses (regex: `[\w.+-]+@[\w-]+\.[\w.]+`) — applies to every record
- Strip personal address lines (room numbers, individual office addresses) 
- Institutional building addresses (e.g. "5640 S University Ave") are **not** stripped

In [106]:
import re

EMAIL_RE = re.compile(r'[\w.+-]+@[\w-]+\.[\w.]+')
# Personal room/office numbers e.g. "Room 340", "Rm. 12", "Office 205"
PERSONAL_ADDR_RE = re.compile(r'\b(?:Room|Rm\.?|Office)\s*\d+\b', re.I)

# Program-level contact addresses that are public and should not be redacted
SAFE_EMAILS = frozenset({
    'applieddatascience-advising@uchicago.edu',
    'applieddatascience-admissions@uchicago.edu',
})

def _replace_email(m: re.Match) -> str:
    return m.group(0) if m.group(0).lower().rstrip('.') in SAFE_EMAILS else '***'

def redact_text(text: str) -> tuple[str, bool]:
    result = EMAIL_RE.sub(_replace_email, text)
    result = PERSONAL_ADDR_RE.sub('***', result)
    return result, result != text

PII_NOTE = (
    "In compliance with responsible AI practices, this page has undergone PII redaction. "
    "Personal email addresses and individual office information have been replaced with ***. "
    "This model is unable to provide the original PII."
)

redacted_indices = []
for i in must_idx:
    changed = False
    r = cleaned_records[i]

    new_atm, ch = redact_text(r.get('all_text_markdown', ''))
    if ch:
        r['all_text_markdown'] = new_atm
        changed = True

    for sec in r.get('structured_sections', []):
        new_md, ch = redact_text(sec['text_markdown'])
        if ch:
            sec['text_markdown'] = new_md
            changed = True

    if changed:
        redacted_indices.append(i)
        existing = r['Note']
        r['Note'] = (PII_NOTE + ' ' + existing).strip() if existing else PII_NOTE

print(f'PII redaction complete. Records modified: {redacted_indices}')
for i in redacted_indices:
    print(f'  [{i}] {cleaned_records[i]["source_file"]}')
    print(f'       Note: {cleaned_records[i]["Note"][:120]}')

PII redaction complete. Records modified: []


## 7. Export as JSONL

### 7.1 Export formatting

In [158]:
DROP_KEYS = {'priority', 'depth', 'special_handling', 'all_text_markdown', 'sections'}

def normalize_url(u):
    if not u:
        return u
    return u.rstrip("/") + "/"

records_for_RAG = [
    {
        **{k: v for k, v in cleaned_records[i].items() if k not in DROP_KEYS},
        'url':           normalize_url(cleaned_records[i].get('url')),
        'canonical_url': normalize_url(cleaned_records[i].get('canonical_url')),
        'breadcrumbs':   ' > '.join(b['text'] for b in cleaned_records[i].get('breadcrumbs') or []),
    }
    for i in must_idx
]

print(f'records_for_RAG: {len(records_for_RAG)} records')
print(f'keys: {list(records_for_RAG[0].keys())}')
records_for_RAG[0]

records_for_RAG: 14 records
keys: ['source_file', 'url', 'canonical_url', 'page_title', 'meta_description', 'breadcrumbs', 'scraped_at', 'structured_sections', 'Note']


{'source_file': 'page_042.json',
 'url': 'https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/',
 'canonical_url': 'https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/',
 'page_title': "Master's in Applied Data Science | DSI",
 'meta_description': "Explore how UChicago's data science master's degree can develop you into a leader in the field by elevating your technical skillset.",
 'breadcrumbs': 'Home > Education > Masters Programs > Ms In Applied Data Science',
 'scraped_at': '2026-04-19T01:41:58.015608+00:00',
 'structured_sections': [{'section_index': 0,
   'text_markdown': 'The University of Chicago’s MS in Applied Data Science program equips you with in-demand expertise and an unparalleled network of global alumni. Take the next step and start your application today.'},
  {'section_index': 1,
   'text_markdown': '## Programs\nChoose from full- and part-time options in our In-Person and Online programs. [Apply 

In [159]:
# Some webpage has alias url
from collections import defaultdict

alias_map = defaultdict(list)
for e in classif["entries"]:
    u, c = normalize_url(e.get("url")), normalize_url(e.get("canonical_url"))
    if u != c:
        if u not in alias_map[c]:
            alias_map[c].append(u)

for r in records_for_RAG:
    extra = alias_map.get(r["canonical_url"], [])
    urls = r["url"] if isinstance(r["url"], list) else [r["url"]]
    r["url"] = list(dict.fromkeys(urls + extra))  

In [160]:
def _plain_english(text: str) -> str:
    """Keep only ASCII letters, hyphens, commas, ampersands, and spaces."""
    cleaned = re.sub(r'[^a-zA-Z ,&-]', ' ', text)
    return re.sub(r' +', ' ', cleaned).strip()

removed_first = []
for r in records_for_RAG:
    ss = r.get('structured_sections', [])
    if not ss:
        continue
    plain = _plain_english(ss[0]['text_markdown'])
    title = r.get('page_title', '')
    if plain and plain.lower() in title.lower():
        removed_first.append((r['source_file'], plain, title))
        r['structured_sections'] = ss[1:]

print(f'Removed redundant first section from {len(removed_first)} records:')
for src, plain, title in removed_first:
    print(f'  {src}')
    print(f'    cleaned : "{plain[:80]}"')
    print(f'    title   : "{title}"')

Removed redundant first section from 8 records:
  page_145.json
    cleaned : "How to Apply"
    title   : "How to Apply | DSI"
  page_146.json
    cleaned : "FAQs"
    title   : "FAQs | DSI"
  page_147.json
    cleaned : "Online Program"
    title   : "Online Program | DSI"
  page_148.json
    cleaned : "Course Progressions"
    title   : "Course Progressions | DSI"
  page_149.json
    cleaned : "In-Person Program"
    title   : "In-Person Program | DSI"
  page_162.json
    cleaned : "Tuition, Fees, & Aid"
    title   : "Tuition, Fees, & Aid | DSI"
  page_163.json
    cleaned : "Our Students"
    title   : "Our Students | DSI"
  page_164.json
    cleaned : "Events & Deadlines"
    title   : "Events & Deadlines | DSI"


In [172]:
records_for_RAG

[{'source_file': 'page_042.json',
  'url': ['https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/'],
  'canonical_url': 'https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/',
  'page_title': "Master's in Applied Data Science | DSI",
  'meta_description': "Explore how UChicago's data science master's degree can develop you into a leader in the field by elevating your technical skillset.",
  'breadcrumbs': 'Home > Education > Masters Programs > Ms In Applied Data Science',
  'scraped_at': '2026-04-19T01:41:58.015608+00:00',
  'structured_sections': [{'section_index': 0,
    'text_markdown': 'The University of Chicago’s MS in Applied Data Science program equips you with in-demand expertise and an unparalleled network of global alumni. Take the next step and start your application today.'},
   {'section_index': 1,
    'text_markdown': '## Programs\nChoose from full- and part-time options in our In-Person and Online prog

### 7.2 Export

In [176]:
OUT_DIR = Path('../../data/cleaned_sections')
OUT_DIR.mkdir(parents=True, exist_ok=True)

for r in records_for_RAG:
    stem = Path(r['source_file']).stem   # e.g. "page_042"
    fname = OUT_DIR / f"{stem}_cleaned.json"
    with open(fname, 'w', encoding='utf-8') as f:
        json.dump(r, f, ensure_ascii=False, indent=2)

print(f'Wrote {len(records_for_RAG)} page files to {OUT_DIR}')

Wrote 14 page files to ..\..\data\cleaned_sections
